In [27]:

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency, mannwhitneyu
from datetime import date

surv   = pd.read_excel(r"..\data\private_dataF.xlsx")

### Data preparation

In [28]:
surv_clean = surv.copy()

# checking the missing values --> result:  NO MISSING VALUES
missing_summary = pd.DataFrame({
    "Missing count": surv_clean.isnull().sum(),
    "Missing percentage": (surv_clean.isnull().mean() * 100).round(2)
})

print(missing_summary)

# Identify all rows with duplicate name and date of birth --> result: NO DUPLICATES
duplicates = surv_clean[
    surv_clean.duplicated(subset=["name", "dob"], keep=False)
].sort_values(["name", "dob"])

print(duplicates)

# Checking the unique values and their frequencies.
categorical_columns = [
    "sex",
    "zip",
    "evote",
    "last_voted",
    "party",
    "marital_status",
    "education",
    "citizenship"
]

for col in categorical_columns:
    if col in surv_clean.columns:
        print(f"\n--- {col} ---")
        print(surv_clean[col].value_counts(dropna=False))


                Missing count  Missing percentage
name                        0                 0.0
sex                         0                 0.0
evote                       0                 0.0
dob                         0                 0.0
zip                         0                 0.0
education                   0                 0.0
citizenship                 0                 0.0
marital_status              0                 0.0
party                       0                 0.0
Empty DataFrame
Columns: [name, sex, evote, dob, zip, education, citizenship, marital_status, party]
Index: []

--- sex ---
sex
Female    100
Male      100
Name: count, dtype: int64

--- zip ---
zip
2400    75
2200    48
2100    40
2300    37
Name: count, dtype: int64

--- evote ---
evote
0    138
1     62
Name: count, dtype: int64

--- party ---
party
Green           127
Red              67
Invalid vote      6
Name: count, dtype: int64

--- marital_status ---
marital_status
Married/separated   

## preparing the demographic attributes
- sex
- zip code
- marital_status
- education
- citizenship
- age

In [29]:
# make sure zip is treated as a category, not a number
surv_clean["zip"] = surv_clean["zip"].astype(str)

# making citizenship being "danish" vs. "non-danish"
surv_clean["citizenship"] = surv_clean["citizenship"].apply(lambda x: "Danish" if x == "Denmark" else "non-danish")

# changing the dob into age and binning it into 3/4 categories
date_then = date(2025, 7, 1)

surv_clean = surv_clean.copy()

surv_clean["age"] = surv_clean["dob"].apply(
    lambda dob: date_then.year - dob.year
    - ((date_then.month, date_then.day) < (dob.month, dob.day))
)

surv_clean["age_group"] = pd.cut(
    surv_clean["age"],
    bins=[18, 30, 45, 65, np.inf],
    labels=["18-29", "30-44", "45-64", "65 or older"],
    right=False,
)


# Remove rows where education is "Not stated"
surv_clean = surv_clean[surv_clean["education"] != "Not stated"].copy()

# education grouping
education_mapping = {
    "Primary education": "Low",
    "Upper secondary education": "Medium",
    "Vocational Education and Training (VET)": "Medium",
    "Short cycle higher education": "High",
    "Vocational bachelors educations": "High",
    "Bachelors programmes": "High",
    "Masters programmes": "High",
    "PhD programmes": "High"
}

surv_clean["education"] = surv_clean["education"].map(education_mapping)

print(surv_clean["education"].value_counts(dropna=False))

surv_clean

education
Medium    85
High      73
Low       38
Name: count, dtype: int64


,name,sex,evote,dob,zip,education,citizenship,marital_status,party,age,age_group
0,"Casas-Marta, Angelica",Female,0,1997-05-24,2100,Medium,Danish,Never married,Green,28,18-29
1,"el-Salahuddin, Rawda",Female,0,1964-01-03,2300,Medium,Danish,Never married,Red,61,45-64
2,"al-Haider, Saahira",Female,1,1976-01-15,2400,High,Danish,Never married,Green,49,45-64
3,"Hansen, Karan",Male,1,1960-04-14,2400,High,Danish,Never married,Red,65,65 or older
4,"Lee, Cody",Male,0,1984-10-17,2100,Medium,Danish,Never married,Green,40,30-44
...,...,...,...,...,...,...,...,...,...,...,...
195,"Rutherford, Joshua",Male,0,1940-05-15,2400,Medium,Danish,Married/separated,Green,85,65 or older
196,"Santee, Michael",Male,1,1964-08-22,2400,Medium,Danish,Married/separated,Green,60,45-64
197,"Elwood, Adrienne",Female,0,1966-04-24,2400,Medium,Danish,Married/separated,Green,59,45-64
198,"al-Jabour, Afaaf",Female,1,1993-01-08,2300,High,Danish,Married/separated,Invalid vote,32,30-44


# Hypothesis
- H0: The choice of voting channel (paper vs. electronic) is independent of [predictor variable]. 
- H1: The choice of voting channel is not independent of [predictor variable].

In [30]:
predictors = {
    "Age group":      "age_group",
    "Gender":         "sex",
    "Nationality":    "citizenship",
    "ZIP code":       "zip",
    "Marital status": "marital_status",
    "Education":      "education"
}

In [33]:
def test_independence(df, col, target="evote"):   # df = surv_clean, col = predictor's column name, i.e. "age_group", target = "evote"
    tab = pd.crosstab(df[col], df[target])            # observed counts --> Counts how many respondents fall into each combination of predictor group and channel.
    chi2, p, dof, expected = chi2_contingency(tab)   # This one line does the whole test and returns four things: expected number of counts if channel and predictor variable were independent (it's MIN expected value); chi2 =the sum of (O − E)² / E, dof = degrees of freedom, p =  the probability of a χ² at least this large if H0 is true
    return tab, dict(chi2=chi2, dof=dof, p=p,
                     min_expected=expected.min(), # expected is the table of E values from chi2_contingency. .min() returns the smallest expected count in the whole table. It tells you how bad the worst cell is.
                     pct_cells_below5=(expected < 5).mean() * 100)   # it tells you what percentage of cells have an expected count below 5.

rows, tables = [], {} # empty list for the results and empty dict for the cross tables
for label, col in predictors.items():
    tab, res = test_independence(surv_clean, col) # Runs the function for this predictor. tab is the crosstab, and res is the results dictionary.
    tables[label] = tab   # Saves the crosstab under its label so you can look at it later (for the % e-vote printout).
    rows.append({"Predictor": label, **res})

summary = pd.DataFrame(rows).set_index("Predictor")
summary["p_holm"] = multipletests(summary["p"], method="holm")[1]   #multipletests takes your 6 raw p-values. It returns a tuple, and [1] selects the adjusted p-values. Holm sorts them from smallest to largest and multiplies by 6, 5, 4, 3, 2, 1
summary["significant (Holm, 0.05)"] = summary["p_holm"] < 0.05 # Creates a True/False column: is the adjusted p below 0.05? All are False here.
print(summary.round(4))

# additional OPTIONAL part
# counts and % e-voting per group
for label, tab in tables.items():
    pct = (tab.div(tab.sum(axis=1), axis=0)[1] * 100).round(1).rename("% e-vote")
    print(f"\n{label}")
    print(pd.concat([tab, pct], axis=1))

                  chi2  dof       p  min_expected  pct_cells_below5  p_holm  \
Predictor                                                                     
Age group       0.9288    3  0.8185        8.0918               0.0   1.000   
Gender          0.0092    1  0.9235       30.1888               0.0   1.000   
Nationality     3.5984    1  0.0578        7.4694               0.0   0.347   
ZIP code        4.2542    3  0.2353       11.5153               0.0   1.000   
Marital status  1.2179    3  0.7487        3.4235              12.5   1.000   
Education       0.2901    2  0.8650       11.8265               0.0   1.000   

                significant (Holm, 0.05)  
Predictor                                 
Age group                          False  
Gender                             False  
Nationality                        False  
ZIP code                           False  
Marital status                     False  
Education                          False  

Age group
            

**Results:** 
- all p-values are above 0.05
- Is the χ² approximation trustworthy? --> Fine everywhere; marital status is borderline but acceptable (one cell, 12.5%)
- What is the p-value after correction (**p_holm**) --> All ≥ 0.347 (so all adjusted p are > 0.05)

**Conclusion**: there is no statistically significant evidence that channel choice depends on any of the six attributes. This is not proof of independence, because with n = 196 only fairly large differences would be detectable.
- **we would not reject any H0**

In [ ]:
# double check for natioanlity, which seems to be borderline
ab = tables["Nationality"]
print("Yates (default):", chi2_contingency(tab)[:2])
print("No correction:  ", chi2_contingency(tab, correction=False)[:2])
print("Fisher exact:   ", stats.fisher_exact(tab))

Yates (default): (np.float64(0.2901192294616146), np.float64(0.864970726576673))
No correction:   (np.float64(0.2901192294616146), np.float64(0.864970726576673))
Fisher exact:    SignificanceResult(statistic=np.float64(0.018062813922676876), pvalue=np.float64(0.8665))
